In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import pairwise_distances

In [2]:
# 1) Load city-level data
city_path = "data/city_safety/city_safety_2023_geo.csv"
cities_df = pd.read_csv(city_path)

print("City columns:", cities_df.columns.tolist())
print(cities_df.head())

# Keep only rows with valid coordinates and city metrics
cities_df = cities_df.dropna(
    subset=["lat", "lon", "crime_index", "safety_index"]
).copy()

# Clean raw country strings: remove spaces, lowercase
cities_df["country"] = cities_df["country"].str.strip()
cities_df["countrynorm"] = cities_df["country"].str.lower()

# Map 2-letter state / province codes (lowercase) → countries
state_to_country = {
    "ak": "united states",
    "ca": "united states",
    "co": "united states",
    "fl": "united states",
    "ga": "united states",
    "hi": "united states",
    "id": "united states",
    "il": "united states",
    "in": "united states",
    "ky": "united states",
    "la": "united states",
    "ma": "united states",
    "mi": "united states",
    "mn": "united states",
    "mo": "united states",
    "nc": "united states",
    "nm": "united states",
    "nv": "united states",
    "ny": "united states",
    "oh": "united states",
    "or": "united states",
    "pa": "united states",
    "tx": "united states",
    "ut": "united states",
    "wa": "united states",
    "wi": "united states",
    "dc": "united states",
    "az": "united states",
    "md": "united states",
    "tn": "united states",
    "ab": "canada",
    "bc": "canada",
}

cities_df["countrynorm"] = cities_df["countrynorm"].replace(state_to_country)

# Handle specific country-name variants (already lowercase here)
name_fixes = {
    "colombia": "colombia",
    "dominican republic": "dominican republic",
    "canada": "canada",
    "thailand": "thailand",
    "moldova": "moldova",
    "georgia": "georgia",          # treat as country; 
    "hong kong (china)": "hong kong",
}
cities_df["countrynorm"] = cities_df["countrynorm"].replace(name_fixes)

print("Unique countrynorm (sample):")
print(cities_df["countrynorm"].unique()[:20])
print("Number of cities:", len(cities_df))

City columns: ['Rank', 'city', 'country', 'crime_index', 'safety_index', 'year', 'city_norm', 'country_norm', 'lat', 'lon']
   Rank          city            country  crime_index  safety_index  year  \
0     1       Caracas          Venezuela         83.6          16.4  2023   
1     2      Pretoria       South Africa         82.0          18.0  2023   
2     3        Durban       South Africa         81.0          19.0  2023   
3     4  Port Moresby   Papua New Guinea         80.7          19.3  2023   
4     5  Johannesburg       South Africa         80.7          19.3  2023   

      city_norm      country_norm        lat         lon  
0       caracas         venezuela  10.506093  -66.914601  
1      pretoria      south africa -25.745928   28.187910  
2        durban      south africa -29.861825   31.009909  
3  port moresby  papua new guinea  -9.474330  147.159950  
4  johannesburg      south africa -26.205000   28.049722  
Unique countrynorm (sample):
['venezuela' 'south africa' 'p

In [3]:
# 2) Load country-level crime data
country_path = "data/crime-rate-by-country-2024.csv"
country_df = pd.read_csv(country_path)

print("Country columns BEFORE rename:", country_df.columns.tolist())

# Rename to clean, short names using your exact column names
country_df = country_df.rename(columns={
    "country": "country",
    "crimeRateByCountry_crimeIndex": "crimeindex",
    "CrimeRate_OverallCriminalityScoreGOCI": "overallcriminalityscore",
    "CrimeRate_CriminalMarketsScore": "criminalmarketsscore",
    "CrimeRate_CriminalActorsScore": "criminalactorsscore",
    "CrimeRate_ResilienceScore": "resiliencescore",
    "CrimeRateSafetyIndex": "safetyindex",
})

print("Country columns AFTER rename:", country_df.columns.tolist())

# Normalize country names for joining
country_df["country"] = country_df["country"].str.strip()
country_df["countrynorm"] = country_df["country"].str.lower()

# Numeric columns we will use
country_numeric_cols = [
    "crimeindex",
    "safetyindex",
    "overallcriminalityscore",
    "criminalmarketsscore",
    "criminalactorsscore",
    "resiliencescore",
]

# Drop rows missing core indices, then median-impute remaining gaps
country_df = country_df.dropna(subset=["crimeindex", "safetyindex"]).copy()

for col in country_numeric_cols:
    if country_df[col].isna().any():
        med = country_df[col].median()
        country_df[col] = country_df[col].fillna(med)

print("Number of countries:", len(country_df))
print(country_df[["country", "countrynorm"]].head())


Country columns BEFORE rename: ['country', 'crimeRateByCountry_crimeIndex', 'CrimeRate_OverallCriminalityScoreGOCI', 'CrimeRate_CriminalMarketsScore', 'CrimeRate_CriminalActorsScore', 'CrimeRate_ResilienceScore', 'CrimeRateSafetyIndex']
Country columns AFTER rename: ['country', 'crimeindex', 'overallcriminalityscore', 'criminalmarketsscore', 'criminalactorsscore', 'resiliencescore', 'safetyindex']
Number of countries: 140
         country    countrynorm
0          India          india
1          China          china
2  United States  united states
3      Indonesia      indonesia
4       Pakistan       pakistan


In [4]:
# 3) Prepare country feature subset for join
country_join_cols = [
    "countrynorm",
    "crimeindex",
    "safetyindex",
    "overallcriminalityscore",
    "criminalmarketsscore",
    "criminalactorsscore",
    "resiliencescore",
]

country_feat_df = country_df[country_join_cols].rename(columns={
    "crimeindex": "country_crimeindex",
    "safetyindex": "country_safetyindex",
    "overallcriminalityscore": "country_overallcriminality",
    "criminalmarketsscore": "country_criminalmarkets",
    "criminalactorsscore": "country_criminalactors",
    "resiliencescore": "country_resilience",
})

# 4) Left join: add country features onto city rows
cities_v2 = cities_df.merge(
    country_feat_df,
    on="countrynorm",
    how="left",
)

# Check unmatched countries
unmatched = cities_v2[cities_v2["country_crimeindex"].isna()]["country"].unique()
print("Unmatched countries:", unmatched)

# Median fill for any remaining missing country features
country_feat_cols = [
    "country_crimeindex",
    "country_safetyindex",
    "country_overallcriminality",
    "country_criminalmarkets",
    "country_criminalactors",
    "country_resilience",
]
for col in country_feat_cols:
    med = cities_v2[col].median()
    cities_v2[col] = cities_v2[col].fillna(med)

print(cities_v2[[
    "city", "country", "countrynorm",
    "crime_index", "country_crimeindex", "country_safetyindex"
]].head())
print("Final number of cities:", len(cities_v2))


Unmatched countries: ['Colombia' 'Dominican Republic' 'AB' 'Canada' 'BC' 'Thailand' 'Moldova'
 'Georgia']
           city           country       countrynorm  crime_index  \
0       Caracas         Venezuela         venezuela         83.6   
1      Pretoria      South Africa      south africa         82.0   
2        Durban      South Africa      south africa         81.0   
3  Port Moresby  Papua New Guinea  papua new guinea         80.7   
4  Johannesburg      South Africa      south africa         80.7   

   country_crimeindex  country_safetyindex  
0                82.1                 17.9  
1                75.5                 24.5  
2                75.5                 24.5  
3                80.4                 19.6  
4                75.5                 24.5  
Final number of cities: 416


In [5]:
# 5) Feature matrix + target
feature_cols_v2 = [
    "lat", "lon",                          # coordinates
    "country_crimeindex",
    "country_safetyindex",
    "country_overallcriminality",
    "country_criminalmarkets",
    "country_criminalactors",
    "country_resilience",
]

X_v2 = cities_v2[feature_cols_v2].values
y_v2 = cities_v2["crime_index"].values   # city-level crime index

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2, y_v2, test_size=0.2, random_state=42
)

# Standardize features for Euclidean distance
scaler_v2 = StandardScaler()
X_train_v2_scaled = scaler_v2.fit_transform(X_train_v2)
X_test_v2_scaled = scaler_v2.transform(X_test_v2)

X_train_v2_scaled.shape, X_test_v2_scaled.shape


((332, 8), (84, 8))

In [6]:
def eval_knn_v2(k, weights):
    model = KNeighborsRegressor(
        n_neighbors=k,
        weights=weights,
        metric="minkowski",   # Euclidean in scaled feature space
    )
    model.fit(X_train_v2_scaled, y_train_v2)
    y_pred = model.predict(X_test_v2_scaled)
    mae = mean_absolute_error(y_test_v2, y_pred)
    rmse = mean_squared_error(y_test_v2, y_pred) ** 0.5
    r2 = r2_score(y_test_v2, y_pred)
    return model, mae, rmse, r2

ks = [3, 5, 10, 20]
weight_opts = ["uniform", "distance"]

results_v2 = []
best_model_v2 = None
best_rmse = float("inf")

for k in ks:
    for w in weight_opts:
        model, mae, rmse, r2 = eval_knn_v2(k, w)
        results_v2.append({
            "k": k,
            "weights": w,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
        })
        if rmse < best_rmse:
            best_rmse = rmse
            best_model_v2 = model

results_v2_df = pd.DataFrame(results_v2).sort_values("RMSE")
results_v2_df


,k,weights,MAE,RMSE,R2
5,10,distance,8.693708,11.076722,0.612010
3,5,distance,8.704227,11.121022,0.608900
1,3,distance,8.640266,11.140599,0.607522
0,3,uniform,8.914683,11.207637,0.602784
7,20,distance,8.972301,11.276555,0.597884
2,5,uniform,9.244524,11.504059,0.581495
4,10,uniform,9.437619,11.594347,0.574900
6,20,uniform,10.236667,12.358340,0.517032


In [7]:
def build_feature_vector_v2(lat, lon, country_name):
    # Look up country risk features
    row = country_df[country_df["country"].str.lower() == country_name.strip().lower()]
    if row.empty:
        raise ValueError(f"Country '{country_name}' not found in country risk data")
    r = row.iloc[0]
    feat = np.array([[
        lat,
        lon,
        r["crimeindex"],
        r["safetyindex"],
        r["overallcriminalityscore"],
        r["criminalmarketsscore"],
        r["criminalactorsscore"],
        r["resiliencescore"],
    ]])
    return scaler_v2.transform(feat)

def predict_crime_knn_v2(lat, lon, country_name, model=None):
    if model is None:
        model = best_model_v2
    x = build_feature_vector_v2(lat, lon, country_name)
    return float(model.predict(x)[0])

def predict_with_radius_check(lat, lon, country_name, model=None, max_radius_km=500):
    if model is None:
        model = best_model_v2

    x_query = build_feature_vector_v2(lat, lon, country_name)

    # Use raw lat/lon to find distance to nearest city
    coords = cities_v2[["lat", "lon"]].values
    dists_deg = pairwise_distances([[lat, lon]], coords, metric="euclidean")[0]
    km_per_degree = 111.0
    dists_km = dists_deg * km_per_degree

    if np.all(dists_km > max_radius_km):
        return None  # no reasonable nearby cities

    return float(model.predict(x_query)[0])


In [9]:
# Best KNN config
results_v2_df.head()

,k,weights,MAE,RMSE,R2
5,10,distance,8.693708,11.076722,0.612010
3,5,distance,8.704227,11.121022,0.608900
1,3,distance,8.640266,11.140599,0.607522
0,3,uniform,8.914683,11.207637,0.602784
7,20,distance,8.972301,11.276555,0.597884


In [8]:
# Example prediction: reuse first city (should be close to its crime_index)
sample = cities_v2.iloc[0]
pred = predict_crime_knn_v2(sample["lat"], sample["lon"], sample["country"])
sample["crime_index"], pred


(83.6, 73.90591469000303)

## v2 results

best result yielded this after adding in country features ---- still need to fix data alighnment issue

Reduces MAE by about 1 point.

Reduces RMSE by about 1 point.

Increases R² from ~0.54 to ~0.61

In [1]:
import ipynbname

nb_path = ipynbname.path()
nb_name = nb_path.name 
nb_name

import sys
from pathlib import Path

# PDF 
!"{sys.executable}" -m nbconvert --to pdf "{nb_name}"

[NbConvertApp] Converting notebook Geo_KNN_v2.ipynb to pdf
[NbConvertApp] Writing 57279 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | b had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 56706 bytes to Geo_KNN_v2.pdf
